# CrimeLens — train CNN + LSTM models (Google Colab / Kaggle)
This notebook trains:
1. **1D-CNN classifier** to estimate the most likely *recorded crime category* from time, location and context features.
2. **LSTM sequence model** to estimate the next-hour count of recorded incidents for each spatial grid, using the preceding 24 hourly bins.

These are educational baselines trained on the supplied synthetic/observational dataset. Incident-only data does **not** contain reliable non-crime exposure labels, population, reporting rates or verified outcomes; therefore outputs are not calibrated real-world crime probabilities and must not be used to label people or make law-enforcement decisions. Use time-based validation to reduce leakage.

In [ ]:
# Colab: upload data.csv when prompted, or edit DATA_PATH for Kaggle
import os, json, random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED=42
np.random.seed(SEED); tf.random.set_seed(SEED)
DATA_PATH = '/content/data.csv'
if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded.keys()))
df=pd.read_csv(DATA_PATH)
df['Timestamp']=pd.to_datetime(df['Timestamp'], errors='coerce')
df=df.dropna(subset=['Timestamp','Latitude','Longitude','Crime_Category']).copy()
df=df.sort_values('Timestamp')
df['Hour']=df['Timestamp'].dt.hour
df['Month']=df['Timestamp'].dt.month
df['DayOfWeek']=df['Timestamp'].dt.dayofweek
df['Hour_sin']=np.sin(2*np.pi*df['Hour']/24)
df['Hour_cos']=np.cos(2*np.pi*df['Hour']/24)
df['Month_sin']=np.sin(2*np.pi*df['Month']/12)
df['Month_cos']=np.cos(2*np.pi*df['Month']/12)
# Keep category-like context encoded into numeric features
for col in ['District','Location_Type','Spatial_Grid_Cell']:
    df[col]=df[col].fillna('Unknown').astype(str)
df['District_code']=pd.factorize(df['District'])[0]
df['Location_code']=pd.factorize(df['Location_Type'])[0]
df['Grid_code']=pd.factorize(df['Spatial_Grid_Cell'])[0]
FEATURES=['Latitude','Longitude','Hour_sin','Hour_cos','Month_sin','Month_cos','DayOfWeek','District_code','Location_code','Grid_code']
# Chronological split
split=int(len(df)*0.8)
train=df.iloc[:split].copy(); test=df.iloc[split:].copy()
scaler=StandardScaler()
X_train=scaler.fit_transform(train[FEATURES])
X_test=scaler.transform(test[FEATURES])
le=LabelEncoder()
y_train=le.fit_transform(train['Crime_Category'].astype(str))
# Only categories known during training can be evaluated
known=test['Crime_Category'].astype(str).isin(le.classes_)
test_known=test.loc[known]
X_test_known=scaler.transform(test_known[FEATURES])
y_test=le.transform(test_known['Crime_Category'].astype(str))
os.makedirs('models',exist_ok=True)


In [ ]:
# 1D CNN: recorded crime-category classifier
cnn=keras.Sequential([
    layers.Input(shape=(len(FEATURES),1)),
    layers.Conv1D(32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(len(le.classes_), activation='softmax')
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.fit(X_train[...,None], y_train, epochs=12, batch_size=64, validation_split=0.15,
        callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=1)
if len(y_test):
    pred=np.argmax(cnn.predict(X_test_known[...,None], verbose=0),axis=1)
    print(classification_report(y_test,pred,target_names=le.classes_,zero_division=0))
cnn.save('models/crime_cnn.keras')


In [ ]:
# LSTM: next-hour incident-count estimate from prior 24 hours per spatial grid
hourly=(df.set_index('Timestamp').groupby('Spatial_Grid_Cell').resample('h').size()
        .rename('count').reset_index().sort_values(['Spatial_Grid_Cell','Timestamp']))
SEQUENCE=24
Xseq=[]; yseq=[]; seq_grid=[]
for grid, g in hourly.groupby('Spatial_Grid_Cell'):
    values=g['count'].to_numpy(dtype=np.float32)
    if len(values)<=SEQUENCE: continue
    for i in range(SEQUENCE,len(values)):
        Xseq.append(values[i-SEQUENCE:i])
        yseq.append(values[i])
        seq_grid.append(grid)
Xseq=np.array(Xseq,dtype=np.float32).reshape(-1,SEQUENCE,1)
yseq=np.array(yseq,dtype=np.float32)
# Chronological order follows grid series construction; use last 20% for holdout
cut=max(1,int(len(Xseq)*0.8))
Xtr,Xte=Xseq[:cut],Xseq[cut:]
ytr,yte=yseq[:cut],yseq[cut:]
# Scale counts using training statistics only
count_mean=float(ytr.mean()) if len(ytr) else 0.0
count_std=float(ytr.std()) if len(ytr) and float(ytr.std())>1e-6 else 1.0
Xtr_s=(Xtr-count_mean)/count_std; Xte_s=(Xte-count_mean)/count_std
ytr_s=(ytr-count_mean)/count_std; yte_s=(yte-count_mean)/count_std
lstm=keras.Sequential([
    layers.Input(shape=(SEQUENCE,1)),
    layers.LSTM(32, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(16),
    layers.Dense(16,activation='relu'),
    layers.Dense(1)
])
lstm.compile(optimizer='adam',loss='mse',metrics=['mae'])
lstm.fit(Xtr_s,ytr_s,epochs=15,batch_size=64,validation_split=0.15,
         callbacks=[keras.callbacks.EarlyStopping(patience=3,restore_best_weights=True)],verbose=1)
if len(Xte):
    forecast=np.maximum(0,lstm.predict(Xte_s,verbose=0).ravel()*count_std+count_mean)
    print('LSTM next-hour count MAE:', round(float(mean_absolute_error(yte,forecast)),3))
lstm.save('models/crime_lstm.keras')


In [ ]:
# Save the preprocessing metadata required by the app
metadata={
 'features': FEATURES,
 'district_mapping': {str(v):int(i) for i,v in enumerate(pd.factorize(df['District'])[1])},
 'location_mapping': {str(v):int(i) for i,v in enumerate(pd.factorize(df['Location_Type'])[1])},
 'grid_mapping': {str(v):int(i) for i,v in enumerate(pd.factorize(df['Spatial_Grid_Cell'])[1])},
 'crime_classes': le.classes_.tolist(),
 'scaler_mean': scaler.mean_.tolist(),
 'scaler_scale': scaler.scale_.tolist(),
 'sequence_length': SEQUENCE,
 'count_mean': count_mean,
 'count_std': count_std,
 'training_rows': int(len(df)),
 'note': 'Historical incident dataset; predictions are not calibrated crime probabilities.'
}
with open('models/metadata.json','w') as f: json.dump(metadata,f,indent=2)
hourly.to_csv('models/hourly_counts.csv',index=False)
print('Saved models/ and metadata. Download the models folder and add it to the GitHub repository.')


## Download artifacts
Run this cell to zip the trained models and download them. Then upload the files inside the generated `models` folder to the `models/` folder in your GitHub repo. Keep the dataset and model provenance documented.

In [ ]:
import shutil
shutil.make_archive('crimelens_models','zip','models')
try:
    from google.colab import files
    files.download('crimelens_models.zip')
except Exception:
    print('Created crimelens_models.zip in the notebook working directory; download it from the Kaggle output/files panel.')
